In [ ]:
import pandas as pd

# Q1: Construct Knowledge Base
fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"}
]

# Example for roll number ending in 23 (2 % 3 = general, 3 % 3 = billing)
custom_entries = [
    {"question": "how to contact support", "answer": "Email support@example.com.", "keywords": "contact support help email", "category": "general"},
    {"question": "where to download receipt", "answer": "Go to Settings > Invoices.", "keywords": "receipt invoice download fee", "category": "billing"}
]

df = pd.DataFrame(fixed_entries + custom_entries)

In [ ]:
# Q2: Initial Hypothesis Scoring
def score_hypothesis(query, df):
    query_words = set(query.lower().split())
    scores = []
    for _, row in df.iterrows():
        kw_set = set(row['keywords'].lower().split())
        q_set = set(row['question'].lower().split())
        score = len(query_words.intersection(kw_set)) * 2 + len(query_words.intersection(q_set))
        scores.append(score)
    
    df_result = df.copy()
    df_result['score'] = scores
    return df_result[df_result['score'] > 0].sort_values(by='score', ascending=False)

In [ ]:
# Q3: same_category function
def same_category(category_name, df):
    return df[df['category'] == category_name][['question', 'category']]

In [ ]:
# Q4: Add keyword and export to CSV
roll_number = "102103423"
df.loc[0, 'keywords'] += " subscription"
df.to_csv(f"{roll_number}_faq_data.csv", index=False)

In [ ]:
# Q5: Groupby counts
print("FAQ Count Per Category:")
print(df.groupby('category').size())
print("\n")

In [ ]:
# Q6: Enhanced Scoring Function with Tie Handling
def score_hypothesis_with_ties(query, df):
    query_words = set(query.lower().split())
    scores = [len(query_words.intersection(set(row['keywords'].lower().split()))) for _, row in df.iterrows()]
    
    df_scored = df.copy()
    df_scored['score'] = scores
    max_score = df_scored['score'].max()
    
    if max_score == 0:
        return pd.DataFrame()
    
    matches = df_scored[df_scored['score'] == max_score]
    if len(matches) > 1:
        print(f"--- TIE DETECTED ({len(matches)} matches with score {max_score}) ---")
    else:
        print(f"--- SINGLE BEST MATCH (score {max_score}) ---")
        
    return matches[['question', 'answer', 'keywords', 'category', 'score']]

# Demonstrations
print(score_hypothesis_with_ties("fee", df))
print(score_hypothesis_with_ties("reset password", df))